# nitid 1: Inference with pretrained models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nitidio/nitid/blob/main/examples/notebooks/01_inference.ipynb)

nitid runs object detection, instance segmentation and semantic segmentation
with models from the DETR family, through one API. This notebook:

1. Detects objects in an image with a pretrained COCO model
2. Runs the same prediction from the command line
3. Segments object instances with masks
4. Exports the detection model

Training on your own data is in
[2: Training and evaluation](02_training.ipynb).

## Setup

**Google Colab:** the cell below installs nitid from PyPI. Inference runs on
the CPU runtime; a GPU runtime is faster.

**Local:** from a nitid checkout, install the dependencies and start Jupyter:

```bash
uv sync
uv run --with jupyterlab jupyter lab
```

Outside a checkout, `pip install nitid` is enough. Images used below are taken
from the checkout, or downloaded from GitHub when there is none, and every
output is written under `runs/notebooks/`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nitid"], check=True)


def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "dfine").is_dir():
            return candidate
    return None


REPO_ROOT = find_repo_root(Path.cwd().resolve())
ROOT = REPO_ROOT or Path.cwd()
WORK_DIR = ROOT / "runs" / "notebooks"
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)  # downloaded checkpoints and datasets/ live here


def repo_file(path: str) -> Path:
    """Return a file from the nitid repository, downloading it outside a checkout."""
    local = ROOT / path
    if not local.exists():
        local.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(f"https://raw.githubusercontent.com/Nitidio/nitid/main/{path}", local)
    return local


print(f"Colab: {IN_COLAB}")
print(f"nitid checkout: {REPO_ROOT}")
print(f"Outputs go to: {WORK_DIR}")

In [ ]:
from nitid import NITID

## 1. Object detection

`NITID("model1m", task="detect")` downloads the official COCO checkpoint on
first use and keeps it in the working directory. The trailing letter selects
the size: `model1s`, `model1m`, `model1l` or `model1x`.

In [ ]:
image_path = repo_file("docs/assets/demo/source/ski.jpg")

model = NITID("model1m", task="detect")
result = model.predict(str(image_path), conf=0.5)[0]

print(result)
for label, score, box in zip(result.boxes.cls, result.boxes.conf, result.boxes.xyxy):
    print(f"{result.names[int(label)]:>8}  {score:.2f}  {box.round().tolist()}")

Boxes are already in pixel coordinates of the original image. `save()` draws
them with their labels:

In [ ]:
from IPython.display import Image, display

detect_path = WORK_DIR / "ski_detect.jpg"
result.save(str(detect_path))
display(Image(filename=str(detect_path), width=800))

The result also converts to other formats, for example
`result.pandas()` (a DataFrame), `result.to_json()` or
`result.crop(save_dir=...)` (one image per box).

## 2. The same prediction from the command line

```bash
nitid predict model=model1m task=detect source=docs/assets/demo/source/ski.jpg conf=0.5 save=true
```

The annotated image is saved under `runs/detect/`. From a notebook, run the
CLI through the current Python interpreter:

In [ ]:
cli = [sys.executable, "-m", "nitid.cli"]
subprocess.run(
    [
        *cli,
        "predict",
        "model=model1m",
        "task=detect",
        f"source={image_path}",
        "conf=0.5",
        "save=true",
        f"project={WORK_DIR / 'cli'}",
        "name=detect",
        "exist_ok=true",
    ],
    check=True,
)

## 3. Instance segmentation

The same API with `task="segment"` loads the matching COCO mask checkpoint and
returns one mask per detected object.

In [ ]:
image_path = repo_file("docs/assets/demo/source/watchmaker.jpg")

segment_model = NITID("model1l", task="segment")
result = segment_model.predict(str(image_path), conf=0.3)[0]

print(result)
print("masks:", tuple(result.masks.data.shape))  # [N, H, W]
print("first polygon:", len(result.masks.xyn[0]), "normalized points")

segment_path = WORK_DIR / "watchmaker_segment.jpg"
result.save(str(segment_path))
display(Image(filename=str(segment_path), width=800))

### Semantic segmentation

`task="semantic"` predicts one class per pixel. There are no pretrained
semantic weights: `NITID("model1s", task="semantic")` starts from the
instance-segmentation checkpoint and has to be
[fine-tuned](https://github.com/Nitidio/nitid/blob/main/docs/fine_tuning.md) on your own masks before its
predictions mean anything.

## 4. Export

The same checkpoint exports to ONNX, OpenVINO, TorchScript and TensorRT.
Exported graphs include the postprocessor, so they return
`(labels, boxes, scores)`. TorchScript needs no extra dependency:

In [ ]:
exported = model.export(
    format="torchscript", output=WORK_DIR / "model1m.torchscript", exist_ok=True
)
print(f"{exported}: {exported.stat().st_size / 1e6:.1f} MB")

ONNX needs a runtime such as `onnxruntime`, and OpenVINO needs
`pip install "nitid[openvino]"`. See the [export guide](https://github.com/Nitidio/nitid/blob/main/docs/export.md)
for every format and option.

## Next

- [2: Training and evaluation](02_training.ipynb): fine-tune on a dataset, validate and export
- [Quickstart](https://github.com/Nitidio/nitid/blob/main/docs/quickstart.md) and [API reference](https://github.com/Nitidio/nitid/blob/main/docs/api_reference.md)